In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 4 - Week 7
# --------------------------------------------------
# The Week 7 .npy files contain the original data
# plus Weeks 1-6 exactly once.
#
# Strategy:
# - fit an ARD Matern GP to all accumulated observations
# - identify the current best observation automatically
# - derive candidate-search widths from fitted lengthscales
# - generate local, wider and global candidates
# - compare EI and UCB before selecting Week 7

In [2]:
X = np.load("function4/initial_inputs.npy")
Y = np.load("function4/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 4

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (36, 4)
Y shape: (36,)

Current best observed input: [0.357812 0.420904 0.424244 0.430762]
Current best observed output: 0.601059871344695


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(4) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
2.46**2 * Matern(length_scale=[1.62, 1.29, 1.29, 1.37], nu=2.5) + WhiteKernel(noise_level=0.00063)


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [1.62379636 1.28961132 1.29349551 1.3741527 ]
Normalised inverse-lengthscale sensitivity: [0.21293982 0.26812025 0.26731512 0.25162481]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.015,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.04,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1 0.1 0.1 0.1]
Wider search widths: [0.2 0.2 0.2 0.2]


In [6]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(50000, 4)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 4)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 4)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 80000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.008]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 79999


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 EI = 0.03643354 

xi=0.001 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 EI = 0.03622175 

xi=0.005 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 EI = 0.0353841 

xi=0.01 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 EI = 0.03435833 

xi=0.02 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 EI = 0.0323765 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.36103789 0.43285023 0.42719443 0.42629656] 
 mean = 0.379486 
 std = 0.263344 
 UCB = 0.40582 

beta=0.25 
 candidate = [0.36103789 0.43285023 0.42719443 0.42629656] 
 mean = 0.379486 
 std = 0.263344 
 UCB = 0.445322 

beta=0.5 
 candidate = [0.33509331 0.4385294  0.42560279 0.42134542] 
 mean = 0.366573 
 std = 0.289563 
 UCB = 0.511354 

beta=1.0 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 UCB = 0.661961 

beta=1.5 
 candidate = [0.33524044 0.4414815  0.43811063 0.42398236] 
 mean = 0.359644 
 std = 0.302317 
 UCB = 0.81312 



In [12]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.36103789 0.43285023 0.42719443 0.42629656]
mean = 0.37948568115190007
std = 0.2633441770534702


In [13]:
# --------------------------------------------------
# Final Function 4 Week 7 selection
# --------------------------------------------------
#
# The GP automatically identified the current best observation and generated
# candidates using widths derived from the fitted ARD lengthscales.
#
# EI consistently preferred a nearby candidate with a lower predicted mean
# but higher uncertainty, indicating a stronger exploration component.
#
# The highest predicted GP mean was found at
# [0.361038, 0.432850, 0.427194, 0.426297].
#
# UCB with both beta=0.1 and beta=0.25 independently selected the same
# candidate. This gives stronger evidence that the recommendation is not
# dependent on a single acquisition setting.
#
# I therefore use low-exploration UCB with beta=0.1 for the Week 7 query.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 4 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 4 candidate:
[0.36103789 0.43285023 0.42719443 0.42629656]

Predicted mean: 0.37948568115190007
Predicted std: 0.2633441770534702
UCB: 0.40582009885724707

Portal format:
0.361038-0.432850-0.427194-0.426297
